In [1]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

In [2]:
power_df = pd.read_csv('/Users/sam/Desktop/StationLevelPowerForecasting/data/power_df_2008-2406.csv')

In [3]:
power_df['date'] = pd.to_datetime(power_df['recordTimestamp'], utc=True)
power_df['date'] = power_df['date'].dt.tz_convert('America/Los_Angeles')
power_df['date'] = power_df['date'].dt.tz_localize(None)

power_df['year'] = power_df['date'].dt.year
power_df['month'] = power_df['date'].dt.month
power_df['hour'] = power_df['date'].dt.hour

power_df['totalPower'] = power_df['totalPower'] / 1000

In [4]:
HOURS_LOOKAHEAD = 4
READINGS_PER_HOUR = 12

power_df['maxPower'] = power_df['totalPower'].rolling(window=HOURS_LOOKAHEAD * READINGS_PER_HOUR, min_periods=1).apply(lambda x: x.max(), raw=True).shift(-47)

In [5]:
power_df = power_df[['totalPower', 'numberOfActiveSessions', 'weekNumber', 'isWeekend', 'isHoliday', 'hour', 'maxPower']]
power_df

,totalPower,numberOfActiveSessions,weekNumber,isWeekend,isHoliday,hour,maxPower
0,0.0,0.0,34,False,False,18,0.0
1,0.0,0.0,34,False,False,18,0.0
2,0.0,0.0,34,False,False,18,0.0
3,0.0,0.0,34,False,False,18,0.0
4,0.0,0.0,34,False,False,18,0.0
...,...,...,...,...,...,...,...
400994,0.0,0.0,24,False,False,2,NaN
400995,0.0,0.0,24,False,False,2,NaN
400996,0.0,0.0,24,False,False,2,NaN
400997,0.0,0.0,24,False,False,2,NaN


In [6]:
power_df = power_df.dropna()

In [7]:
X = power_df[['totalPower', 'numberOfActiveSessions', 'weekNumber', 'isWeekend', 'isHoliday', 'hour']]

# Perform one-hot encoding on the 'hour' column
hour_one_hot = pd.get_dummies(X['hour'], prefix='hour')

# Concatenate the one-hot encoded columns back to the original DataFrame
X = pd.concat([X, hour_one_hot], axis=1)

# Optionally, you can drop the original 'hour' column if you no longer need it
X = X.drop('hour', axis=1)

X

,totalPower,numberOfActiveSessions,weekNumber,isWeekend,isHoliday,hour_0,hour_1,hour_2,hour_3,hour_4,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0.0,0.0,34,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
1,0.0,0.0,34,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,0.0,0.0,34,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,0.0,0.0,34,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,0.0,0.0,34,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400947,0.0,0.0,24,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
400948,0.0,0.0,24,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
400949,0.0,0.0,24,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
400950,0.0,0.0,24,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [8]:
y = power_df['maxPower']

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Assuming X is your DataFrame and y is your target Series

# Step 1: Optionally, split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Initialize the Linear Regression model
model = LinearRegression()

# Step 3: Fit the model to the training data
model.fit(X_train, y_train)

# Step 4: Optionally, make predictions on the test set
y_pred = model.predict(X_test)

# Step 5: Optionally, evaluate the model's performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

# If you want to make predictions on new data
# y_new_pred = model.predict(new_X)


Mean Squared Error: 28.530021830068755
R^2 Score: 0.7292724494362264
